# 🎤 ChuckleNet — XLM-R Stand-Up Laughter Prediction

**Canonical pipeline — May 2026**

| Metric | Best Result |
|--------|------------|
| Validation F1 | 0.7850 |
| Test F1 | 0.8194 |
| Test IoU-F1 | 0.8798 |

**Backbone:** `FacebookAI/xlm-roberta-base`  
**Teacher:** `qwen3:4b` (Ollama)  
**Winner:** weak-label XLM-R with `positive_class_weight=5.0`

## 0. ⚙️ Setup & Mount Drive

In [ ]:
# @title Install dependencies
!pip install -q transformers torch accelerate datasets scikit-learn tqdm

import os, sys, json
from pathlib import Path

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ──
DRIVE_ROOT    = Path('/content/drive/MyDrive/chucklenet')
DATA_DIR      = DRIVE_ROOT / 'data' / 'training' / 'standup_word_level'
EXPERIMENTS_DIR = DRIVE_ROOT / 'experiments'
REPO_DIR      = Path('/content/ChuckleNet')

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Drive root: {DRIVE_ROOT}")
print(f"GPU: {!nvidia-smi -L 2>/dev/null || echo 'No GPU detected'}")

## 1. 📦 Clone ChuckleNet Repo

In [ ]:
# @title Clone repository
%cd /content
!rm -rf ChuckleNet
!git clone https://github.com/Das-rebel/ChuckleNet.git
%cd ChuckleNet

# Install repo-specific deps
!pip install -q -r requirements.txt 2>/dev/null || echo "No requirements.txt"
print(f"Repo at: {os.getcwd()}")
print(f"Files: {len(os.listdir('.'))}")

## 2. 📊 Data: Raw → Word-Level JSONL

Converts raw stand-up transcripts to word-level sequence-labeling examples.
Each word gets a binary label (0=no laughter, 1=laughter trigger).

In [ ]:
# @title Stage 1: Convert raw transcripts to word-level JSONL
!python3 training/convert_standup_raw_to_word_level.py \
  --raw-dir "{DRIVE_ROOT}/data/raw" \
  --dataset-dir "{DATA_DIR}" \
  --split-strategy overlap_safe \
  --context-token-policy clause_lexical_tail \
  --context-tail-tokens 6

# Verify output
for split in ['train', 'valid', 'test']:
    f = DATA_DIR / f'{split}.jsonl'
    if f.exists():
        lines = len(open(f).readlines())
        print(f"  {split}: {lines} examples")
    else:
        print(f"  {split}: MISSING!")

## 3. 🧠 Refine Labels (Ollama Teacher)

Uses a local teacher model to refine weak laughter labels.
**Skip this if you already have refined labels or just want the weak-label baseline.**

In [ ]:
# @title Stage 2 (OPTIONAL): Refine weak labels with teacher
REFINE = False  # @param {type:"boolean"}

if REFINE:
    !pip install -q ollama openai  # for API compat
    
    # Start Ollama if not running (Colab needs manual setup or ngrok tunnel)
    # Option A: Run Ollama locally and expose via ngrok
    # Option B: Use OpenAI-compatible endpoint
    
    TEACHER_MODEL = "qwen3:4b"  # @param {type:"string"}
    TEACHER_ENDPOINT = "http://127.0.0.1:11434/api/generate"  # @param {type:"string"}
    
    !python3 training/refine_weak_labels_nemotron.py \
      --backend ollama \
      --endpoint "{TEACHER_ENDPOINT}" \
      --teacher-model "{TEACHER_MODEL}" \
      --teacher-min-confidence 0.6 \
      --teacher-write-every 25 \
      --teacher-resume
else:
    print("Skipping refinement. Using weak labels directly.")

## 4. 🚀 Train XLM-R Baseline

Trains `xlm-roberta-base` on word-level laughter prediction.
The recommended config uses `positive_class_weight=5.0` and `cross_entropy` loss.

In [ ]:
# @title Stage 3: Train XLM-R
POS_CLASS_WEIGHT = 5.0    # @param {type:"number"}
EPOCHS = 3                # @param {type:"integer"}
BATCH_SIZE = 2            # @param {type:"integer"}
LEARNING_RATE = 2e-5      # @param {type:"number"}
LOSS_TYPE = "cross_entropy"  # @param ["cross_entropy", "focal", "adaptive_focal"]

!python3 training/run_xlmr_standup_pipeline.py \
  --skip-convert \
  --skip-refine \
  --dataset-dir "{DATA_DIR}" \
  --output-dir "{EXPERIMENTS_DIR}/xlmr_weak_pos{POS_CLASS_WEIGHT:.0f}" \
  --model-name FacebookAI/xlm-roberta-base \
  --batch-size {BATCH_SIZE} \
  --eval-batch-size {BATCH_SIZE} \
  --epochs {EPOCHS} \
  --learning-rate {LEARNING_RATE} \
  --classifier-learning-rate 1e-4 \
  --gradient-accumulation-steps 4 \
  --max-length 256 \
  --freeze-encoder-epochs 1 \
  --unfreeze-last-n-layers 4 \
  --loss-type {LOSS_TYPE} \
  --positive-class-weight {POS_CLASS_WEIGHT} \
  --decode-strategy argmax \
  --refined-train-policy weak \
  --split-strategy overlap_safe \
  --prune-best-model-weights

print("\n✅ Training complete!")

## 5. 📈 Evaluate & Metrics

In [ ]:
# @title Evaluate best checkpoint
import glob

# Find latest experiment
exp_dirs = sorted(glob.glob(str(EXPERIMENTS_DIR / 'xlmr_weak_pos*')))
if exp_dirs:
    best = exp_dirs[-1]
    print(f"Evaluating: {best}")
    
    # Run evaluation
    !python3 training/evaluate_saved_xlmr_model.py \
      --model-dir "{best}" \
      --test-data "{DATA_DIR}/test.jsonl" \
      --valid-data "{DATA_DIR}/valid.jsonl" \
      --batch-size 8
    
    # Also run WESR benchmark suite
    !python3 training/evaluate_wesr_benchmark_suite.py \
      --model-dir "{best}" \
      --dataset-dir "{DATA_DIR}" \
      --batch-size 8 2>/dev/null || echo "WESR suite not available"
else:
    print("No experiment directories found!")
    !ls -la "{EXPERIMENTS_DIR}/" 

## 6. 🔬 Autoresearch Loop

Runs autonomous hyperparameter search. Tests variations and promotes
only when a new best is found on the validation set.

In [ ]:
# @title Run Autoresearch Loop
CYCLES = 2  # @param {type:"integer"}

!python3 training/autonomous_research_loop.py \
  --dataset-dir "{DATA_DIR}" \
  --experiments-root "{EXPERIMENTS_DIR}" \
  --promoted-model-path "{EXPERIMENTS_DIR}/promoted_model.json" \
  --max-cycles {CYCLES} \
  --model-name FacebookAI/xlm-roberta-base \
  --epochs 3 \
  --batch-size 2 \
  --split-strategy overlap_safe

print("\n✅ Autoresearch complete!")
print("Check promoted_model.json for best config.")

## 7. 📋 Download Results

Best model and metrics are saved to your Google Drive.

In [ ]:
# @title Show results
import json
from pathlib import Path

promoted = EXPERIMENTS_DIR / 'promoted_model.json'
if promoted.exists():
    with open(promoted) as f:
        data = json.load(f)
    print("🏆 PROMOTED MODEL:")
    print(json.dumps(data, indent=2))
else:
    print("No promoted_model.json yet. Run training first.")

# List all experiment dirs
print("\n📁 All experiments:")
for d in sorted(EXPERIMENTS_DIR.iterdir()):
    if d.is_dir():
        size = sum(f.stat().st_size for f in d.rglob('*') if f.is_file())
        print(f"  {d.name} ({size/1e6:.1f} MB)")

# Download best model
print("\n📦 Your model checkpoints are in:")
print(f"   {EXPERIMENTS_DIR}/")
print("\n💾 Also saved to your Google Drive at:")
print(f"   {DRIVE_ROOT}/")